# Inference of Taxonomic and Functional Composition of Microbial Communities

We will:

1. Download dataset (using web scrapping)
2. Create Manifest file for Qiime2
3. Install dependencies and Qiime2
4. Run Qiime2 analysis


# Clone Github Repo

Start by cloning our repo to be able to store our data orderly

In [13]:
# 1. Clone GitHub repo to Colab
REPO_URL = "https://github.com/hector-ahg/predictive-microbiology-course.git"

!git clone $REPO_URL
%cd predictive-microbiology-course

Cloning into 'predictive-microbiology-course'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 214 (delta 39), reused 52 (delta 18), pack-reused 141 (from 1)
Receiving objects: 100% (214/214), 122.45 MiB | 24.15 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/predictive-microbiology-course/dataset/ibdmdb_hmp2/raw_sequences/predictive-microbiology-course


# Load packages

In [14]:
import requests #The requests module allows you to send HTTP requests using Python
from bs4 import BeautifulSoup #Beautiful Soup is a Python library for pulling data out of HTML and XML files. It provides tools for parsing and navigating the parse tree, making it easy to extract data from web pages.
from urllib.parse import urljoin, urlparse #The urllib.parse module provides functions for manipulating URLs, including joining base URLs with relative URLs to create absolute URLs.
import os #The os module provides a way of using operating system dependent functionality like reading or writing to the file system, handling file paths, and managing directories.
from pathlib import Path #The pathlib allows you to work with file paths in an object-oriented way, making it easier to manipulate and interact with file system paths across different operating systems.
import subprocess
import sys
import numpy as np
import pandas as pd

# Folder structure

In [96]:
# Base directories
BASE_DIR = Path("/content/predictive-microbiology-course").resolve() #project_root

print("Current working directory:", BASE_DIR)

DATA_DIR = (BASE_DIR / "dataset").resolve()
DATA_IBDMDB = (DATA_DIR /"ibdmdb_hmp2").resolve()
PATH_sequences = (DATA_IBDMDB / "raw_sequences").resolve()
PATH_manifest =  (DATA_IBDMDB / "manifests").resolve()
PATH_results = (BASE_DIR / "results").resolve()
QC_DIR = (PATH_results / "quality_control").resolve() # demux files

# Create folders if they does not exist
for d in [
    PATH_sequences,
    PATH_manifest,
    PATH_results,
    QC_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

Current working directory: /content/predictive-microbiology-course


# Downloading IBDMDB Data

This notebook demonstrates how to download raw sequencing data from the Inflammatory Bowel Disease Multi-omics Database (https://ibdmdb.org/) using basic web scraping techniques in Python.

## Goals

1. Become familiar with sending HTTP requests using 'requests'.
2. Learn how to extract data from HTML using 'BeautifulSoup'.

In [16]:
# Download raw files (sequences) from IBDMDB

url= "https://ibdmdb.org/downloads/html/rawfiles_16s_2018-01-08.html"

response = requests.get(url)
print('Status code:', response.status_code) # if 200, the request was successful

Status code: 200


In [17]:
# Parse the HTML content of the page to find the download links for the raw files

soup = BeautifulSoup(response.text, "html.parser")
print(soup) # print the HTML content of the page to understand its structure and identify the relevant elements containing the download links.

<!DOCTYPE html>

<html>
<head>
<meta charset="utf-8"/>
<title>Results</title>
<link href="/static/css/bootstrap-theme.min.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/bootstrap.min.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/ibdmdb.css" rel="stylesheet" type="text/css"/>
<link href="/static/css/mezzanine.css" rel="stylesheet" type="text/css"/>
</head>
<body>
<div class="navbar navbar-inverse navbar-fixed-top" role="navigation">
<div class="container-fluid">
<div class="navbar-header">
<a class="navbar-brand" href="/" style="padding: 0px 10px;">
<img alt="IBDMDB.org" height="50" src="/static/img/gut_cartoon.png" width="50"/>
</a>
<ul class="nav navbar-nav">
<li id="dropdown-menu-home">
<a href="/">Home</a>
</li>
<li id="dropdown-menu-home">
<a href="/results">Download Data</a>
</li>
<li id="dropdown-menu-home">
<a href="/protocols">Protocols</a>
</li>
<li id="dropdown-menu-home">
<a href="/project">Team</a>
</li>
<li id="dropdown-menu-home">
<a hre

In [18]:

links = []
# iterate over the elements of the webpage 'a' finds all anchor tags with an href attribute, which typically represent hyperlinks in HTML. The href attribute contains the URL of the linked resource.
for a in soup.find_all('a', href=True):
    link = a['href'] # extract the URL of the linked resource
    # check if the link ends with '.fastq.gz', which is a common file extension for compressed FASTQ files (sequences), often used for storing biological sequence data.
    if link.endswith('.fastq.gz'):
        full_link = urljoin(url, link) # Convert relative URL to absolute URL
        links.append(full_link)
        print('Download link found:', full_link)

Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206534.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206536.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206538.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206547.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206548.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206561.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206562.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206563.fastq.gz
Download link found: https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/raw/HMP2/16S/2018-01-08/206564.f

In [19]:
# print the current working directory to confirm where the files will be saved
#print(os.getcwd())

# Change the current working directory to the desired output directory
#os.chdir(PATH_sequences)

#print(os.getcwd())

In [20]:
# Download each file from the list of links

n= len(links) # get the total number of download links found on the webpage
#n = 10  # number of files to download (uncomment if you want to download a subset of size n)

for i, url in enumerate(links[:n]): # iterate over the first n links in the list of download links
    response = requests.get(url, stream=True) # send a GET request to the URL to download the file. The stream=True parameter allows you to download the file in chunks, which is useful for large files.
    response.raise_for_status() # check if the request was successful (status code 200). If the request was not successful, this will raise an HTTPError exception.

    filename = os.path.basename(urlparse(url).path) # extract the filename from the URL using urlparse to parse the URL and os.path.basename to get the base name of the file from the path component of the URL.
    filepath = PATH_sequences/filename # create the full file path by joining the current working directory with the filename.

    print(f"Downloading file {i}: {filename}") # print a message indicating which file is being downloaded, including the index of the file in the list and the filename.

    with open(filepath, "wb") as f: # open the file in binary write mode to save the downloaded content. The with statement ensures that the file is properly closed after writing.
        for chunk in response.iter_content(chunk_size=8192): # iterate over the content of the response in chunks of 8192 bytes (8 KB) using the iter_content method. This allows you to download large files without consuming too much memory.
            f.write(chunk) # write each chunk of data to the file until the entire file is downloaded.

    print("File downloaded successfully:", filename)
print("All files downloaded successfully.")

File downloaded successfully: 206534.fastq.gz
File downloaded successfully: 206536.fastq.gz
File downloaded successfully: 206538.fastq.gz
File downloaded successfully: 206547.fastq.gz
File downloaded successfully: 206548.fastq.gz
File downloaded successfully: 206561.fastq.gz
File downloaded successfully: 206562.fastq.gz
File downloaded successfully: 206563.fastq.gz
File downloaded successfully: 206564.fastq.gz
File downloaded successfully: 206569.fastq.gz
All files downloaded successfully.


# Create manifest

In [21]:
# Change working directory
#os.chdir(PATH_sequences)

#print("Now working in:", os.getcwd())

# Collect files
fastq_files = list(PATH_sequences.glob("*.fastq.gz"))

Now working in: /content/predictive-microbiology-course/dataset/ibdmdb_hmp2/raw_sequences/predictive-microbiology-course/dataset/ibdmdb_hmp2/raw_sequences


In [22]:
# Qiime2 requires a manifest of sequences to work
# Build manifest
manifest = pd.DataFrame({
    "sample-id": [f.stem.replace(".fastq", "") for f in fastq_files],
    "absolute-filepath": [str(f.resolve()) for f in fastq_files]
})

# Optional: sort for reproducibility
manifest = manifest.sort_values("sample-id")

# Save manifest file in PATH_manifest
manifest.to_csv(PATH_manifest/ "manifest_in_colab.tsv", sep="\t", index=False)

manifest.head()


,sample-id,absolute-filepath
9,206534,/content/predictive-microbiology-course/datase...
2,206536,/content/predictive-microbiology-course/datase...
0,206538,/content/predictive-microbiology-course/datase...
1,206547,/content/predictive-microbiology-course/datase...
4,206548,/content/predictive-microbiology-course/datase...


In [23]:
# Check manifest file exists
manifest_file = PATH_manifest / "manifest_in_colab.tsv"

if not manifest_file.is_file():
    raise FileNotFoundError("Manifest file not found!")

print("Manifest file found:", manifest_file)
# print number of samples
len(manifest)

Manifest file found: /content/predictive-microbiology-course/dataset/ibdmdb_hmp2/raw_sequences/predictive-microbiology-course/dataset/ibdmdb_hmp2/manifests/manifest_in_colab.tsv


10

# Setup

Installation of Miniconda, Mamba, QIIME2, and their corresponding helper functions.

## Install Miniconda

In [24]:
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -p /opt/conda

PREFIX=/opt/conda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/conda


In [25]:
os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/opt/conda/bin:' + os.environ['PATH']

In [26]:
!conda --version

conda 26.3.2


In [27]:
# Accept Terms of Service
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r


accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


## Install Mamba


In [30]:
#%%capture # to silece warning messages
!conda install -y -n base -c conda-forge mamba

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | done
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: / - \ | / - \ | done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - mamba


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    conda-26.3.2               |  py313h78bf25f_1         1.2 MB  conda-forge
    cpp-expected-1.3.1         |       h171cf75_0          24 KB  conda-forge
    libarchive-3.8.7           |       hb3cce40_0         867 KB
    libcurl-8.20.0             |       hd8fa685_1         491 KB
    libmamba-2.6.1             |       hd28c85e_0         2.7 MB  conda-forge
    libmamba-spdlog-2.6.1      |       hd8d719e_0          20 KB  conda-forge
    libmambapy-2.6.1           | 

In [31]:
!mamba --version

2.6.1


## Install QIMME2

In [32]:
# This command can take up to 40 minutes to complete.
# IMPORTANT: Around 30 minutes into the execution, you will be prompted to type 'Y' to continue the installation.
!mamba env create -n qiime2-amplicon-2024.10 --file https://data.qiime2.org/distro/amplicon/qiime2-amplicon-2024.10-py310-linux-conda.yml

Streaming output truncated to the last 5000 lines.
Extracting  (667)  ⣾  [+] 1m:12.9s
Extracting  (667)  ⣾  [+] 1m:13.0s
Extracting  (667)  ⣾  [+] 1m:13.1s
Extracting  (667)  ⣾  [+] 1m:13.2s
Extracting  (667)  ⣾  [+] 1m:13.3s
Extracting  (667)  ⣾  [+] 1m:13.4s
Extracting  (666)  ⣾  [+] 1m:13.5s
Extracting  (666)  ⣾  [+] 1m:13.6s
Extracting  (666)  ⣾  [+] 1m:13.7s
Extracting  (665)  ⣾  [+] 1m:13.8s
Extracting  (665)  ⣾  [+] 1m:13.9s
Extracting  (665)  ⣾  [+] 1m:14.0s
Extracting  (665)  ⣾  [+] 1m:14.1s
Extracting  (664)  ⣾  [+] 1m:14.2s
Extracting  (664)  ⣾  [+] 1m:14.3s
Extracting  (664)  ⣾  [+] 1m:14.4s
Extracting  (664)  ⣾  [+] 1m:14.5s
Extracting  (664)  ⣾  [+] 1m:14.6s
Extracting  (664)  ⣾  [+] 1m:14.7s
Extracting  (662)  ⣾  [+] 1m:14.8s
Extracting  (662)  ⣾  [+] 1m:14.9s
Extracting  (662)  ⣾  [+] 1m:15.0s
Extracting  (662)  ⣾  [+] 1m:15.1s
Extracting  (661)  ⣾  [+] 1m:15.2s
Extracting  (661)  ⣾  [+] 1m:15.3s
Extracting  (661)  ⣾  [+] 1m:15.4s
Extracting  (661)  ⣾  [+] 1m:15.5s
Extr

In [33]:
# Check installation
!source /opt/conda/bin/activate qiime2-amplicon-2024.10 && qiime info

QIIME is caching your current deployment for improved performance. This may take a few moments and should only happen once per deployment.
System versions
Python version: 3.10.14
QIIME 2 release: 2024.10
QIIME 2 version: 2024.10.1
q2cli version: 2024.10.1

Installed plugins
alignment: 2024.10.0
composition: 2024.10.0
cutadapt: 2024.10.0
dada2: 2024.10.0
deblur: 2024.10.0
demux: 2024.10.0
diversity: 2024.10.0
diversity-lib: 2024.10.0
emperor: 2024.10.0
feature-classifier: 2024.10.0
feature-table: 2024.10.0
fragment-insertion: 2024.10.0
longitudinal: 2024.10.0
metadata: 2024.10.0
phylogeny: 2024.10.0
quality-control: 2024.10.0
quality-filter: 2024.10.0
rescript: 2024.10.0
sample-classifier: 2024.10.0
stats: 0+unknown
taxa: 2024.10.0
types: 2024.10.0
vizard: 0.0.1.dev0
vsearch: 2024.10.0

Application config directory
/opt/conda/envs/qiime2-amplicon-2024.10/var/q2cli

Config
Config Source: /opt/conda/envs/qiime2-amplicon-2024.10/etc/qiime2_config.toml

Getting help
To get help with QIIME 2

## Helper function for QIIME2

In [177]:
# Helper function for running QIIME2 (it does not interfer with PIICRUST2)
def qiime(cmd):
    full_cmd = f"source /opt/conda/bin/activate qiime2-amplicon-2024.10 && {cmd}"
    result = subprocess.run(
        full_cmd,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

## Is QIIME2 working?

In [52]:
qiime("qiime --version")

RUNNING:
mamba run -n qiime2-amplicon-2024.10 qiime --version

STDOUT:
q2cli version 2024.10.1
Run `qiime info` for more version details.


STDERR:


RETURN CODE:
0


0

In [90]:
qiime("qiime tools --help")

RUNNING COMMAND:


    source /opt/conda/bin/activate qiime2-amplicon-2024.10
    qiime tools --help
    

STDOUT:

Usage: qiime tools [OPTIONS] COMMAND [ARGS]...

  Tools for working with QIIME 2 files.

Options:
  --help      Show this message and exit.

Commands:
  cache-create              Create an empty cache at the given location.
  cache-fetch               Fetches an artifact out of a cache into a .qza.
  cache-garbage-collection  Runs garbage collection on the cache at the
                            specified location.
  cache-import              Imports data into an Artifact in the cache under a
                            key.
  cache-remove              Removes a given key from a cache.
  cache-status              Checks the status of the cache.
  cache-store               Stores a .qza in the cache under a key.
  cast-metadata             Designate metadata column types.
  citations                 Print citations for a QIIME 2 result.
  export                    Export 

CompletedProcess(args='\n    source /opt/conda/bin/activate qiime2-amplicon-2024.10\n    qiime tools --help\n    ', returncode=0, stdout='Usage: qiime tools [OPTIONS] COMMAND [ARGS]...\n\n  Tools for working with QIIME 2 files.\n\nOptions:\n  --help      Show this message and exit.\n\nCommands:\n  cache-create              Create an empty cache at the given location.\n  cache-fetch               Fetches an artifact out of a cache into a .qza.\n  cache-garbage-collection  Runs garbage collection on the cache at the\n                            specified location.\n  cache-import              Imports data into an Artifact in the cache under a\n                            key.\n  cache-remove              Removes a given key from a cache.\n  cache-status              Checks the status of the cache.\n  cache-store               Stores a .qza in the cache under a key.\n  cast-metadata             Designate metadata column types.\n  citations                 Print citations for a QIIME 2 res

# QIIME2 analysis

In [178]:
# Sequencing mode
sequencing_mode = "SINGLE-END"
# sequencing_mode = "PAIRED-END"


# Import sequences into QIIME2

if sequencing_mode == "SINGLE-END":

    print("Importing single-end data...")

    res = qiime(f"""
    qiime tools import \
      --type 'SampleData[SequencesWithQuality]' \
      --input-path {PATH_manifest}/manifest_in_colab.tsv \
      --output-path {QC_DIR}/demux.qza \
      --input-format SingleEndFastqManifestPhred33V2
    """)

elif sequencing_mode == "PAIRED-END":

    print("Importing paired-end data...")

    res = qiime(f"""
    qiime tools import \
      --type 'SampleData[PairedEndSequencesWithQuality]' \
      --input-path {PATH_manifest}/manifest_in_colab.tsv \
      --output-path {QC_DIR}/demux.qza \
      --input-format PairedEndFastqManifestPhred33V2
    """)

else:

    raise ValueError(
        "sequencing_mode must be either "
        "'SINGLE-END' or 'PAIRED-END'"
    )


# Verify import succeeded
if res.returncode == 0:

    print("Import completed successfully.")

    print("Output:")
    print(QC_DIR / "demux.qza")

else:

    print("QIIME2 import failed.")

Importing single-end data...
Imported /content/predictive-microbiology-course/dataset/ibdmdb_hmp2/manifests/manifest_in_colab.tsv as SingleEndFastqManifestPhred33V2 to /content/predictive-microbiology-course/results/quality_control/demux.qza

Import completed successfully.
Output:
/content/predictive-microbiology-course/results/quality_control/demux.qza


In [99]:
# Quality summary
qiime(f"""
qiime demux summarize \
  --i-data {QC_DIR}/demux.qza \
  --o-visualization {QC_DIR}/demux-summary.qzv
""")
print("Check visualization of sequence quality: https://view.qiime2.org/")


Saved Visualization to: /content/predictive-microbiology-course/results/quality_control/demux-summary.qzv

Check visualization of sequence quality: https://view.qiime2.org/


## Primer information


This dataset follows the Earth Microbiome Project (EMP)
Human Microbiome Project (HMP) V4 16S protocol.

Typical V4 primers:

Forward primer (515F): GTGCCAGCMGCCGCGGTAA

Reverse primer (806R): GACTACHVGGGTWTCTAAT

Public datasets may already have primers removed.
Inspect FASTQ files before running Cutadapt.

In [187]:
# Quickly check if primers were removed or not yet

!zcat {PATH_sequences}/*.fastq.gz | head -40

@206534_0 M70287:115:000000000-B59YV:1:1101:18877:1845 1:N:0: orig_bc=CACTACGCTAGA new_bc=CACTACGCTAGA bc_diffs=0
TACGGAGGATCCGAGCGTTATCCGGATTTATTGGGTTTAAAGGGAGCGTAGGTGGACTGGTAAGTCAGTTGTGAAAGTTTGCGGCTCAACCGTAAAATTGCAGTTGATACTGTCAGTCTTGAGTACAGTAGAGGTGGGCGGAATTCGTGGTGTAGCGGTGAAATGCTTAGATATCACGAAGAACTCCGATTGCGAAGGCAGCTCACTGGACTGCAACTGACACTGATGCTCGAAAGTGTGGGTATCAAACAGG
+
>1>FFFF91B>BF?AEEGGGGGHGGGGGHHHHHHGGFGHHHHGG/EEEGHGGBGG01GF;GGHFHFGHHHHHHHGHBGHGCBHF>FEGFFH?ECGGBFHHFBFFHFHHHHHHHFHGHGHGHHGHHEHHHHHHGHHHGGHHHHGFHGFGHHHGHHHHHHHHHHHHHHHHHHHHHHHGCEEHHGGGBFGHGGGGGGHHHHHHHHFFHGEHGHHGFGGBHHGGFFEHFHEFFFFGGGGFCGGGGFFDFFFFFA>>1
@206534_1 M70287:115:000000000-B59YV:1:1101:19413:1985 1:N:0: orig_bc=CACTACGCTAGA new_bc=CACTACGCTAGA bc_diffs=0
TACGTAGGTGGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCGCGCAGGTGGTTTAATAAGTCTGATGTGAAAGCCCACGGCTCAACCGTGGAGGGTCATTGGAAACTGTTAAACTTGAGTGCAGGAGAGAAAAGTGGAATTCCTAGTGTAGCGGTGAAATGCGTAGAGATTAGGAGGAACACCAGTGGCGAAGGCGGCTTTTTGGCCTGTAACTGACACTGAGGCGCGAAAGCGTGGGGAGCAAACAGG
+
>>>BFF

#### Remove primers if necessary

The IBDMDB dataset has the primers removed. Thus, we do not need to runn the following code.

In [ ]:
# Primer trimming with Cutadapt

# Define primers
FORWARD_PRIMER = "GTGCCAGCMGCCGCGGTAA"
REVERSE_PRIMER = "GGACTACHVGGGTWTCTAAT"  # only used for paired-end


# SINGLE-END
if sequencing_mode == "SINGLE-END":

    print("Removing primers from single-end reads...")

    res = qiime(f"""
    qiime cutadapt trim-single \
      --i-demultiplexed-sequences {QC_DIR}/demux.qza \
      --p-front {FORWARD_PRIMER} \
      --p-error-rate 0.1 \
      --p-overlap 3 \
      --o-trimmed-sequences {QC_DIR}/demux-trimmed.qza \
      --verbose
    """)


# PAIRED-END
elif sequencing_mode == "PAIRED-END":

    print("Removing primers from paired-end reads...")

    res = qiime(f"""
    qiime cutadapt trim-paired \
      --i-demultiplexed-sequences {QC_DIR}/demux.qza \
      --p-front-f {FORWARD_PRIMER} \
      --p-front-r {REVERSE_PRIMER} \
      --p-error-rate 0.1 \
      --p-overlap 3 \
      --o-trimmed-sequences {QC_DIR}/demux-trimmed.qza \
      --verbose
    """)

else:

    raise ValueError(
        "sequencing_mode must be either "
        "'SINGLE-END' or 'PAIRED-END'"
    )


# Verify trimming succeeded
if res.returncode == 0:

    print("Primer trimming completed successfully.")

    print("Trimmed output:")
    print(QC_DIR / "demux-trimmed.qza")

else:

    print("Cutadapt trimming failed.")

In [ ]:
# Visualize trimmed read quality

qiime(f"""
qiime demux summarize \
  --i-data {QC_DIR}/demux-trimmed.qza \
  --o-visualization {QC_DIR}/demux-trimmed-summary.qzv
""")

print("Check visualization:")
print("https://view.qiime2.org/")

## DADA2 with truncation sweep

In [100]:
# DADA2 truncation sweep

INPUT = f"{QC_DIR}/demux.qza"
THREADS = 2
TRUNCS = [180] # Modify or add truncation values. Feel free to test multiple values!

for T in TRUNCS:
    OUTDIR = f"{PATH_results}/dada2_trunc_{T}"

    print(f"Running denoise-single with trunc-len={T}")

    os.makedirs(OUTDIR, exist_ok=True)

    qiime(
    f"qiime dada2 denoise-single "
    f"--i-demultiplexed-seqs {INPUT} "
    f"--p-trunc-len {T} "
    f"--p-n-threads {THREADS} "
    f"--o-table {OUTDIR}/table.qza "
    f"--o-representative-sequences {OUTDIR}/rep-seqs.qza "
    f"--o-denoising-stats {OUTDIR}/stats.qza "
    f"--verbose"
    )

print("All DADA2 runs completed!")

# Select truncation

SELECTED_T = 180
DADA2_RESULT_DIR = f"{PATH_results}/dada2_trunc_{SELECTED_T}"

print(f"Using results from truncation length: {SELECTED_T}")

# 4. Visualize denoising stats

qiime(f"""
qiime metadata tabulate \
  --m-input-file {DADA2_RESULT_DIR}/stats.qza \
  --o-visualization {DADA2_RESULT_DIR}/stats.qzv
""")

Running denoise-single with trunc-len=180
R version 4.3.3 (2024-02-29) 
DADA2: 1.30.0 / Rcpp: 1.0.13.1 / RcppParallel: 5.1.9 
2) Filtering ..........
3) Learning Error Rates
44906400 total bases in 249480 reads from 10 samples will be used for learning the error rates.
4) Denoise samples 
..........
5) Remove chimeras (method = consensus)
6) Report read numbers through the pipeline
7) Write output
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_dada.R --input_directory /tmp/qiime2/root/data/90834528-4486-4650-b015-2a0c8b9a7e87/data --output_path /tmp/tmpy5jwxsur/output.tsv.biom --output_track /tmp/tmpy5jwxsur/track.tsv --filtered_directory /tmp/tmpy5jwxsur --truncation_length 180 --trim_left 0 --max_expected_errors 2.0 --truncation_quality_score 2 --max_length Inf --pooling_method independe

0

In [101]:
# Filter samples (low frequency)

qiime(
    f"qiime feature-table filter-samples "
    f"--i-table {DADA2_RESULT_DIR}/table.qza "
    f"--p-min-frequency {2000} "
    f"--o-filtered-table {DADA2_RESULT_DIR}/table-filtered-samples.qza"
)

# ================================
# Filter low-abundance features
# ================================

qiime(
    f"qiime feature-table filter-features "
    f"--i-table {DADA2_RESULT_DIR}/table-filtered-samples.qza "
    f"--p-min-frequency {25} "
    f"--o-filtered-table {DADA2_RESULT_DIR}/table-filtered-final.qza"
)

# ================================
# Summarize
# ================================

qiime(
    f"qiime feature-table summarize "
    f"--i-table {DADA2_RESULT_DIR}/table-filtered-final.qza "
    f"--o-visualization {DADA2_RESULT_DIR}/summary-filtered-final.qzv"
)

print("Filtering and summarization complete!")

Saved FeatureTable[Frequency] to: /content/predictive-microbiology-course/results/dada2_trunc_180/table-filtered-samples.qza

Saved FeatureTable[Frequency] to: /content/predictive-microbiology-course/results/dada2_trunc_180/table-filtered-final.qza

Saved Visualization to: /content/predictive-microbiology-course/results/dada2_trunc_180/summary-filtered-final.qzv

Filtering and summarization complete!


## Export for PICRUSt2

In [153]:
# Define export directory
EXPORT_FOR_PICRUST_DIR = (
    Path(DADA2_RESULT_DIR)  / "export_for_picrust2"
).resolve()

EXPORT_FOR_PICRUST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(f"Exporting files to: {EXPORT_FOR_PICRUST_DIR}")

# Export Feature table
qiime(
    f"qiime tools export \
  --input-path {DADA2_RESULT_DIR}/table-filtered-final.qza \
  --output-path {EXPORT_FOR_PICRUST_DIR}/table_export"
)

# Export Representative sequences
qiime(
    f"qiime tools export \
  --input-path {DADA2_RESULT_DIR}/rep-seqs.qza \
  --output-path {EXPORT_FOR_PICRUST_DIR}/seqs_export"
)

print("Export for PICRUSt2 complete.")

Exporting files to: /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2
Exported /content/predictive-microbiology-course/results/dada2_trunc_180/table-filtered-final.qza as BIOMV210DirFmt to directory /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export

Exported /content/predictive-microbiology-course/results/dada2_trunc_180/rep-seqs.qza as DNASequencesDirectoryFormat to directory /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/seqs_export

Export for PICRUSt2 complete.


# Download Silva Bayesian classifier

In [161]:
# Classifier directory
CLASSIFIER_DIR = (
    BASE_DIR /
    "pipeline" / "qiime2" / "classifiers"
).resolve()

CLASSIFIER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Output file
classifier_path = (
    CLASSIFIER_DIR /
    "silva-138-99-nb-classifier.qza"
)

# Working URL
url = (
    "https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza"
)

# Download
subprocess.run(
    [
        "wget",
        "-O",
        str(classifier_path),
        url
    ],
    check=True
)

print("Classifier downloaded:")
print(classifier_path)

print("File exists:")
print(classifier_path.exists())

Classifier downloaded:
/content/predictive-microbiology-course/pipeline/qiime2/classifiers/silva-138-99-nb-classifier.qza
File exists:
True


In [163]:
# feature-classifier

qiime(
    f"qiime feature-classifier classify-sklearn \
    --i-reads {DADA2_RESULT_DIR}/rep-seqs.qza \
    --i-classifier {CLASSIFIER_DIR}/silva-138-99-nb-classifier.qza \
    --p-n-jobs 2 \
    --o-classification {DADA2_RESULT_DIR}/taxa.qza"
    )

qiime(
    f"qiime tools export \
  --input-path {DADA2_RESULT_DIR}/taxa.qza \
  --output-path {DADA2_RESULT_DIR}/taxa_export"
)



Saved FeatureData[Taxonomy] to: /content/predictive-microbiology-course/results/dada2_trunc_180/taxa.qza

Exported /content/predictive-microbiology-course/results/dada2_trunc_180/taxa.qza as TSVTaxonomyDirectoryFormat to directory /content/predictive-microbiology-course/results/dada2_trunc_180/taxa_export



0

# Trees

In [195]:
qiime(
    f"qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences {DADA2_RESULT_DIR}/rep-seqs.qza \
    --output-dir {DADA2_RESULT_DIR}/tree"
)



Saved FeatureData[AlignedSequence] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/alignment.qza
Saved FeatureData[AlignedSequence] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/masked_alignment.qza
Saved Phylogeny[Unrooted] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/tree.qza
Saved Phylogeny[Rooted] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/rooted_tree.qza



CompletedProcess(args='source /opt/conda/bin/activate qiime2-amplicon-2024.10 && qiime phylogeny align-to-tree-mafft-fasttree     --i-sequences /content/predictive-microbiology-course/results/dada2_trunc_180/rep-seqs.qza     --output-dir /content/predictive-microbiology-course/results/dada2_trunc_180/tree', returncode=0, stdout='Saved FeatureData[AlignedSequence] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/alignment.qza\nSaved FeatureData[AlignedSequence] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/masked_alignment.qza\nSaved Phylogeny[Unrooted] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/tree.qza\nSaved Phylogeny[Rooted] to: /content/predictive-microbiology-course/results/dada2_trunc_180/tree/rooted_tree.qza\n', stderr='')

# Diversity

In [196]:
DIV_DIR = (
    Path(DADA2_RESULT_DIR)  / "diversity"
).resolve()

DIV_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [200]:
DIV_DIR.mkdir(parents=True, exist_ok=True)

import shutil

if DIV_DIR.exists():
    shutil.rmtree(DIV_DIR)
    print("Removed existing diversity directory")


qiime(
    f"qiime diversity core-metrics-phylogenetic \
  --i-table {DADA2_RESULT_DIR}/table.qza \
  --i-phylogeny {DADA2_RESULT_DIR}/tree/rooted_tree.qza \
  --p-sampling-depth 10000 \
  --m-metadata-file {METADATA_DIR}/metadata.tsv \
  --output-dir {DIV_DIR}"
)

Removed existing diversity directory
Saved FeatureTable[Frequency] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/evenness_vector.qza
Saved DistanceMatrix to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/unweighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/weighted_unifrac_distance_matrix.qza
Saved DistanceMatrix to: /cont

CompletedProcess(args='source /opt/conda/bin/activate qiime2-amplicon-2024.10 && qiime diversity core-metrics-phylogenetic   --i-table /content/predictive-microbiology-course/results/dada2_trunc_180/table.qza   --i-phylogeny /content/predictive-microbiology-course/results/dada2_trunc_180/tree/rooted_tree.qza   --p-sampling-depth 10000   --m-metadata-file /content/predictive-microbiology-course/dataset/ibdmdb_hmp2/metadata/metadata.tsv   --output-dir /content/predictive-microbiology-course/results/dada2_trunc_180/diversity', returncode=0, stdout='Saved FeatureTable[Frequency] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/rarefied_table.qza\nSaved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/faith_pd_vector.qza\nSaved SampleData[AlphaDiversity] to: /content/predictive-microbiology-course/results/dada2_trunc_180/diversity/observed_features_vector.qza\nSaved SampleData[AlphaDiversity] to: /content/p

# Boxplots


In [202]:
qiime(
    f"qiime taxa barplot \
  --i-table {DADA2_RESULT_DIR}/table-filtered-final.qza \
  --i-taxonomy {DADA2_RESULT_DIR}/taxa.qza \
  --m-metadata-file {METADATA_DIR}/metadata.tsv \
  --o-visualization {DADA2_RESULT_DIR}/taxa_barplot.qzv"
)


Saved Visualization to: /content/predictive-microbiology-course/results/dada2_trunc_180/taxa_barplot.qzv



CompletedProcess(args='source /opt/conda/bin/activate qiime2-amplicon-2024.10 && qiime taxa barplot   --i-table /content/predictive-microbiology-course/results/dada2_trunc_180/table-filtered-final.qza   --i-taxonomy /content/predictive-microbiology-course/results/dada2_trunc_180/taxa.qza   --m-metadata-file /content/predictive-microbiology-course/dataset/ibdmdb_hmp2/metadata/metadata.tsv   --o-visualization /content/predictive-microbiology-course/results/dada2_trunc_180/taxa_barplot.qzv', returncode=0, stdout='Saved Visualization to: /content/predictive-microbiology-course/results/dada2_trunc_180/taxa_barplot.qzv\n', stderr='')

# Install PICRUSt2

To ensure compatibility and avoid conflicts with QIIME2's dependencies, PICRUSt2 will be installed in a separate `conda` environment named `picrust2`. This allows both bioinformatics tools to run smoothly in the same notebook.

In [203]:
# This may take around 30 minutes to run

# Create picrust2 environment
!mamba create -n picrust2 -c conda-forge -c bioconda picrust2=2.6.2 #2.5.0

# Check installation
!source /opt/conda/bin/activate picrust2 && picrust2 --version

Found conda-prefix at '/opt/conda/envs/picrust2'. Overwrite?: [y/N] y
warning  libmamba 'repo.anaconda.com', a commercial channel hosted by Anaconda.com, is used.
    
warning  libmamba Please make sure you understand Anaconda Terms of Services.
    
warning  libmamba See: https://legal.anaconda.com/policies/en/
[+] 0.0s
[+] 0.1s
pkgs/r/linux-64 (c..  ⣾  [+] 0.2s
bioconda/noarch (c..  ⣾  
pkgs/main/linux-64..  ⣾  
pkgs/r/noarch (che..  ⣾  ⚠ Shard Index for bioconda/linux-64 not available, falling back to flat repodata
Using Flat Repodata for bioconda/linux-64                                                 ✔ Done (0.4 sec)
⚠ Shard Index for bioconda/noarch not available, falling back to flat repodata
Using Flat Repodata for bioconda/noarch                                                   ✔ Done (1.1 sec)
⚠ Shard Index for pkgs/main/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/linux-64                                                ✔ Done (0.3

## Helper function for PICRUST2

In [206]:
import subprocess

# Helper function to run PICRUSt2 commands
def picrust(cmd):

    # Remove extra whitespace/newlines
    cmd = " ".join(cmd.split())

    full_cmd = f"mamba run -n picrust2 {cmd}"

    print("RUNNING:")
    print(full_cmd)

    result = subprocess.run(
        full_cmd,
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print("\nSTDOUT:")
    print(result.stdout)

    if result.stderr:
        print("\nSTDERR:")
        print(result.stderr)

    print("\nRETURN CODE:")
    print(result.returncode)

    return result

## Is PICRUST2 working?

In [204]:
!mamba run -n picrust2 picrust2_pipeline.py --help

usage: picrust2_pipeline.py
       [-h]
       -s
       PATH
       -i
       PATH
       -o
       PATH
       [-p PROCESSES]
       [-t epa-ng|sepp]
       [-r1 PATH]
       [-r2 PATH]
       [--in_traits IN_TRAITS]
       [--custom_trait_tables_ref1 PATH]
       [--custom_trait_tables_ref2 PATH]
       [--marker_gene_table_ref1 PATH]
       [--marker_gene_table_ref2 PATH]
       [--pathway_map MAP]
       [--reaction_func MAP]
       [--no_pathways]
       [--regroup_map ID_MAP]
       [--no_regroup]
       [--stratified]
       [--max_nsti FLOAT]
       [--min_reads INT]
       [--min_samples INT]
       [-m {mp,emp_prob,pic,scp,subtree_average}]
       [-e EDGE_EXPONENT]
       [--min_align MIN_ALIGN]
       [--skip_minpath]
       [--no_gap_fill]
       [--coverage]
       [--per_sequence_contrib]
       [--wide_table]
       [--skip_norm]
       [--remove_intermediate]
       [--verbose]
       [-v]

Wrapper for full PICRUSt2 pipeline to be run with two different domains. This 

In [205]:
# Check if the picrust2 executable exists in the environment's bin directory
!ls -l /opt/conda/envs/picrust2/bin/picrust2_pipeline.py

-rwxrwxr-x 1 root root 18734 May 15 04:53 /opt/conda/envs/picrust2/bin/picrust2_pipeline.py


# Prepare DADA2 output for PICRUSt2

In [207]:
# This script prepares the data from DADA2 to PICRUSt2
# and runs functional prediction.

# Make folders
METADATA_DIR = (DATA_IBDMDB / "metadata").resolve()
SEQUENCE_EXP_DIR = (EXPORT_FOR_PICRUST_DIR /"seqs_export").resolve()
TABLE_EXP_DIR = (EXPORT_FOR_PICRUST_DIR / "table_export").resolve()

for d in [
    SEQUENCE_EXP_DIR,
    METADATA_DIR,
    QC_DIR,
    TABLE_EXP_DIR
]:
    d.mkdir(parents=True, exist_ok=True)


picrust(f"""
biom convert \
  -i {TABLE_EXP_DIR}/feature-table.biom \
  -o {TABLE_EXP_DIR}/feature-table.tsv \
  --to-tsv
""")

# Convert BIOM table to HDF5 format
picrust(f"""
biom convert \
  -i {TABLE_EXP_DIR}/feature-table.biom \
  -o {TABLE_EXP_DIR}/table.biom \
  --to-hdf5
""")



RUNNING:
mamba run -n picrust2 biom convert -i /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/feature-table.biom -o /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/feature-table.tsv --to-tsv

STDOUT:


RETURN CODE:
0
RUNNING:
mamba run -n picrust2 biom convert -i /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/feature-table.biom -o /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/table.biom --to-hdf5

STDOUT:


RETURN CODE:
0


CompletedProcess(args='mamba run -n picrust2 biom convert -i /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/feature-table.biom -o /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/table.biom --to-hdf5', returncode=0, stdout='', stderr='')

# Run PICRUST2

In [210]:
!mamba list -n picrust2 | grep -E "epa|sepp|hmmer"

  epa-ng                     0.3.8         h077b44d_7             bioconda   
  hmmer                      3.1b2         3                      bioconda   
  sepp                       4.4.0         py39_0                 bioconda   


In [213]:
# Remove previous directories to ensure picrust2 works
!rm -rf {PATH_results}/picrust2_out

# Run PICRUSt2 # crashes using processes = 2 and epa-ng, thus use processes = 1 and sepp
# Using a new, unique output directory name to avoid conflicts
res = picrust(f"""
picrust2_pipeline.py \
  -s {SEQUENCE_EXP_DIR}/dna-sequences.fasta \
  -i {TABLE_EXP_DIR}/table.biom \
  -o {PATH_results}/picrust2_out \
  -p 1 \
  --stratified \
  --placement_tool sepp
""")


if res.returncode == 0:
    print("PICRUSt2 completed!")
else:
    print("PICRUSt2 pipeline failed.")

RUNNING:
mamba run -n picrust2 picrust2_pipeline.py -s /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/seqs_export/dna-sequences.fasta -i /content/predictive-microbiology-course/results/dada2_trunc_180/export_for_picrust2/table_export/table.biom -o /content/predictive-microbiology-course/results/picrust2_out -p 1 --stratified --placement_tool sepp


KeyboardInterrupt: 

Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 160, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 


# Save your files in YOUR Google drive (recommended)

In [186]:
from google.colab import drive
from pathlib import Path
import shutil

# Toggle Drive export
SAVE_TO_DRIVE = True

# Drive folder name
drive_folder = "RESULTS_FROM_QIIME2_PICRUST2_IN_COLAB"

def mount_drive():

    drive.mount(
        "/content/driveDL",
        force_remount=True
    )

    project_root_drive = Path(
        f"/content/driveDL/MyDrive/{drive_folder}"
    )

    project_root_drive.mkdir(
        parents=True,
        exist_ok=True
    )

    return project_root_drive


# Optional Drive export
if SAVE_TO_DRIVE:

    project_root_drive = mount_drive()

    drive_export_path = (
        project_root_drive /
        "results"
    )

    shutil.copytree(
        PATH_results,
        drive_export_path,
        dirs_exist_ok=True
    )

    print(f"Files copied to:\n{drive_export_path}")

Mounted at /content/driveDL
Files copied to:
/content/driveDL/MyDrive/RESULTS_FROM_QIIME2_PICRUST2_IN_COLAB/results
